# Judge Compound Bias — DAY 2: Real Dataset, Full 2³ Run
**This notebook is complete and self-contained** — it replaces yesterday's notebook entirely. All of yesterday's fixes are already applied (tuned `apply_length`, `INCLUDE_LENGTH=True`). Fresh Colab runtime, **T4 GPU**, run cells top to bottom.

Flow: install → transforms (final versions) → load judge → **load MT-Bench + close-pair filter** → validate length on real data → full run (resumable) → analysis → save to Drive.

In [1]:
#@title Cell 1 — Install + GPU check
!pip install -q transformers accelerate bitsandbytes statsmodels pandas numpy datasets
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.1 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4


In [2]:
#@title Cell 2 — Transformations (FINAL versions - length fix already applied)
import re

def apply_position(resp_a, resp_b, level):
    """level 0: (A,B); level 1: (B,A). Returns (first, second, first_slot_is_a)."""
    if level == 0:
        return resp_a, resp_b, True
    return resp_b, resp_a, False

def apply_formatting(text, level):
    """level 1: same sentences re-wrapped as a markdown list under a heading."""
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    return "### Explanation\n" + "\n".join(f"- {s}" for s in sentences)

_FILLER_CLOSERS = [" This point is worth keeping in mind.", " That is the key idea here.",
                   " This is the essential reasoning."]

def apply_length(text, level):
    """FINAL (validated day 1): appends a neutral discourse closer after each
    sentence. No restatement, no new content. Ratio ~1.3-1.6 on short text,
    lower on long text (that's fine - report the measured ratio)."""
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    return " ".join(f"{s}{_FILLER_CLOSERS[i % len(_FILLER_CLOSERS)]}"
                    for i, s in enumerate(sentences))

_CONTENT_WORD_RE = re.compile(r"\b[a-z]{4,}\b")
_FILLER_WORDS = set()
for _p in _FILLER_CLOSERS:
    _FILLER_WORDS.update(re.findall(r"[a-z]{4,}", _p.lower()))

def validate_length_preservation(original, expanded):
    orig = set(_CONTENT_WORD_RE.findall(original.lower()))
    exp = set(_CONTENT_WORD_RE.findall(expanded.lower()))
    added = exp - orig - _FILLER_WORDS
    return {"length_ratio": round(len(expanded)/max(len(original),1), 2),
            "new_content_words": sorted(added), "clean": len(added) == 0}

print("transforms ready (final)")

transforms ready (final)


In [9]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [1]:
#@title Cell 3 — Load the judge (Qwen2.5-7B-Instruct, 4-bit) + smoke test (~6 min)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
PROMPT_VERSION = "v1"

_JUDGE_TEMPLATE = """You are an impartial evaluator. You will be shown a question and two candidate answers, labelled [Answer 1] and [Answer 2]. Decide which answer is better, judging only on correctness, relevance, and helpfulness. Ignore differences in length, formatting, and ordering.

Question:
{question}

[Answer 1]
{first}

[Answer 2]
{second}

Respond with EXACTLY one line in the form:
Verdict: 1
or
Verdict: 2
or
Verdict: tie"""

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model.eval()

def _generate(prompt, max_new_tokens=16):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _parse_verdict(raw):
    m = re.search(r"verdict\s*[:\-]?\s*(1|2|tie)", raw, re.IGNORECASE)
    if not m:
        m = re.search(r"\b(1|2|tie)\b", raw, re.IGNORECASE)
    if not m:
        return "UNPARSED"
    v = m.group(1).lower()
    return "TIE" if v == "tie" else v

def judge_pair(question, first, second):
    raw = _generate(_JUDGE_TEMPLATE.format(question=question, first=first, second=second))
    return {"slot_verdict": _parse_verdict(raw), "raw": raw, "prompt_version": PROMPT_VERSION}

def slot_to_ab(slot_verdict, first_slot_is_a):
    if slot_verdict in ("TIE", "UNPARSED"):
        return slot_verdict
    if slot_verdict == "1":
        return "A" if first_slot_is_a else "B"
    return "B" if first_slot_is_a else "A"

print(judge_pair("Why does ice float on water?",
                 "Water expands when it freezes, lowering its density.",
                 "Ice is less dense than liquid water, so it floats."))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [3]:
#@title Cell 4 — Load MT-Bench human judgments + CLOSE-PAIR filter
# Close pairs = where surface bias can actually tip the verdict (day-1 lesson:
# zero flips on lopsided pairs). We keep pairs where human raters DISAGREED or
# voted tie (close), plus a minority of unanimous pairs as robustness anchors.
from datasets import load_dataset
import pandas as pd

ds = load_dataset("lmsys/mt_bench_human_judgments")
print("splits:", list(ds.keys()))
split = "human" if "human" in ds else list(ds.keys())[0]
raw = ds[split].to_pandas()
print("columns:", list(raw.columns))
print("n rows:", len(raw))

# ---- keep turn 1 only (single-turn Q->A keeps the judging prompt clean) ----
if "turn" in raw.columns:
    raw = raw[raw["turn"] == 1]

def _extract(row):
    """conversation_X = list of {'role','content'} messages. Turn 1:
    [0]=user question, [1]=assistant answer."""
    try:
        conv_a, conv_b = row["conversation_a"], row["conversation_b"]
        q = conv_a[0]["content"]
        a = conv_a[1]["content"]
        b = conv_b[1]["content"]
        return pd.Series({"question": q, "response_a": a, "response_b": b})
    except Exception:
        return pd.Series({"question": None, "response_a": None, "response_b": None})

extracted = raw.apply(_extract, axis=1)
raw = pd.concat([raw.reset_index(drop=True), extracted.reset_index(drop=True)], axis=1)
raw = raw.dropna(subset=["question", "response_a", "response_b"])

# ---- length cap: keep runtime sane on a T4 and prompts within context ----
MAX_CHARS = 2200
raw = raw[(raw.response_a.str.len() < MAX_CHARS) & (raw.response_b.str.len() < MAX_CHARS)
          & (raw.question.str.len() < 600)]
print("after turn/length filters:", len(raw))

# ---- close vs anchor via human vote agreement per unique pair ----
key = ["question_id", "model_a", "model_b"] if "question_id" in raw.columns else ["question", "model_a", "model_b"]
def classify(group):
    votes = group["winner"].tolist()
    n_tie = sum(v == "tie" for v in votes)
    n_a = sum(v == "model_a" for v in votes)
    n_b = sum(v == "model_b" for v in votes)
    if n_tie > 0 or (n_a > 0 and n_b > 0):
        cat = "close"          # raters split or tied -> surface cues can matter
    else:
        cat = "anchor"         # unanimous -> clear-cut robustness control
    r = group.iloc[0]
    return pd.Series({"question": r["question"], "response_a": r["response_a"],
                      "response_b": r["response_b"], "category": cat,
                      "n_votes": len(votes)})

pairs = raw.groupby(key, as_index=False).apply(classify, include_groups=False).reset_index(drop=True)
print(pairs.category.value_counts())

# ---- final item set: up to 260 close + 40 anchors ----
close = pairs[pairs.category == "close"].head(260)
anchor = pairs[pairs.category == "anchor"].head(40)
final = pd.concat([close, anchor]).reset_index(drop=True)

items = [{"item_id": f"mtb_{i}_{r.category}", "question": r.question,
          "response_a": r.response_a, "response_b": r.response_b,
          "category": r.category}
         for i, r in final.iterrows()]
print(f"ITEMS: {len(items)} total = {len(close)} close + {len(anchor)} anchor")
print("\nEXAMPLE CLOSE PAIR:\nQ:", items[0]["question"][:200])
print("A:", items[0]["response_a"][:200])
print("B:", items[0]["response_b"][:200])

README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/gpt4_pair-00000-of-00001-c0b431264a(…): reconstructing file:   0%|          |  0.00B /  650kB            

data/gpt4_pair-00000-of-00001-c0b431264a(…): downloading bytes:           |  0.00B            

data/human-00000-of-00001-25f49108187592(…): reconstructing file:   0%|          |  0.00B /  739kB            

data/human-00000-of-00001-25f49108187592(…): downloading bytes:           |  0.00B            

Generating gpt4_pair split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Generating human split:   0%|          | 0/3355 [00:00<?, ? examples/s]

splits: ['gpt4_pair', 'human']
columns: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn']
n rows: 3355
after turn/length filters: 1272
category
anchor    610
close     281
Name: count, dtype: int64
ITEMS: 300 total = 260 close + 40 anchor

EXAMPLE CLOSE PAIR:
Q: Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.
A: Aloha! I recently had the pleasure of embarking on a trip to the beautiful island of Hawaii, and let me tell you, the cultural experiences and must-see attractions did not disappoint.

First on my lis
B: Here is a draft travel blog post about a recent trip to Hawaii:

Aloha from the Hawaiian Islands! I recently had the opportunity to spend 10 days exploring the beautiful Hawaiian archipelago, soaking 


In [4]:
#@title Cell 4b — re-derive STRONG anchors (>=2 votes, unanimous), then run them
# `pairs` and `items` must be in memory from Cell 4.
strong_pool = pairs[(pairs.category == "anchor") & (pairs.n_votes >= 2)]
print("strong anchor pool available:", len(strong_pool))

strong_anchor_items = [
    {"item_id": f"mtbA{i}_anchor2", "question": r.question,
     "response_a": r.response_a, "response_b": r.response_b,
     "category": "anchor2"}
    for i, r in strong_pool.head(40).reset_index(drop=True).iterrows()
]
print("strong anchors selected:", len(strong_anchor_items))

# new item_ids -> resume logic won't skip them; appends to the same JSONL
run_experiment(strong_anchor_items, judge_name="qwen2.5-7b",
               out_path="results/raw_qwen.jsonl")

strong anchor pool available: 163
strong anchors selected: 40


NameError: name 'run_experiment' is not defined

In [5]:
#@title Reload + verify the new anchors landed
import pandas as pd, json
df = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
print("total records:", len(df))
print(df.groupby("category").item_id.nunique())

FileNotFoundError: [Errno 2] No such file or directory: 'results/raw_qwen.jsonl'

In [5]:
#@title Cell 5 — Validate length transform on REAL responses (gate before the run)
# Day-1 validation was on toy text; this is the distribution that matters.
import numpy as np
ratios, dirty = [], 0
for it in items[:20]:
    for resp in (it["response_a"], it["response_b"]):
        v = validate_length_preservation(resp, apply_length(resp, 1))
        ratios.append(v["length_ratio"])
        if not v["clean"]:
            dirty += 1
            print("NOT CLEAN:", it["item_id"], v["new_content_words"][:10])
print(f"ratio: mean {np.mean(ratios):.2f}, min {min(ratios):.2f}, max {max(ratios):.2f}")
print(f"dirty: {dirty}/40  (want 0)")
print("\nEYEBALL (first real expansion):\n", apply_length(items[0]["response_a"], 1)[:600])
# If dirty>0: usually the sentence splitter hit code blocks/lists in an MT-Bench
# answer. Either drop those items or paste the output to Claude to fix.

ratio: mean 1.41, min 1.04, max 2.81
dirty: 0/40  (want 0)

EYEBALL (first real expansion):
 Aloha! This point is worth keeping in mind. I recently had the pleasure of embarking on a trip to the beautiful island of Hawaii, and let me tell you, the cultural experiences and must-see attractions did not disappoint. That is the key idea here. First on my list was a visit to the Polynesian Cultural Center. This is the essential reasoning. This interactive experience immerses you in the cultures of the Pacific Islands, from the intricate dances of the Maori people of New Zealand to the fire knife dancing of Samoa. This point is worth keeping in mind. The center also features a canoe pageant


In [7]:
#@title Cell 6 — Factorial harness (2^3 = 8 conditions, INCLUDE_LENGTH=True)
import itertools, json, time
from pathlib import Path

INCLUDE_LENGTH = True   # length validated day 1 + cell 5 above

def _factor_levels():
    f = {"position": [0, 1], "formatting": [0, 1]}
    if INCLUDE_LENGTH:
        f["length"] = [0, 1]
    return f

def _all_conditions():
    f = _factor_levels(); keys = list(f)
    for combo in itertools.product(*[f[k] for k in keys]):
        yield dict(zip(keys, combo))

def _build_condition(resp_a, resp_b, cond):
    a = apply_formatting(resp_a, cond.get("formatting", 0))
    b = apply_formatting(resp_b, cond.get("formatting", 0))
    a = apply_length(a, cond.get("length", 0))
    b = apply_length(b, cond.get("length", 0))
    return apply_position(a, b, cond.get("position", 0))

def run_item(item, judge_name):
    base_cond = {k: 0 for k in _factor_levels()}
    first, second, fia = _build_condition(item["response_a"], item["response_b"], base_cond)
    res = judge_pair(item["question"], first, second)
    baseline_ab = slot_to_ab(res["slot_verdict"], fia)
    for cond in _all_conditions():
        first, second, fia = _build_condition(item["response_a"], item["response_b"], cond)
        res = judge_pair(item["question"], first, second)
        v = slot_to_ab(res["slot_verdict"], fia)
        flipped = (baseline_ab in ("A", "B")) and (v in ("A", "B")) and v != baseline_ab
        yield {"item_id": item["item_id"], "judge": judge_name,
               "category": item.get("category", "na"),
               "position": cond.get("position", 0), "formatting": cond.get("formatting", 0),
               "length": cond.get("length", 0), "first_slot_is_a": fia,
               "slot_verdict": res["slot_verdict"], "verdict_ab": v,
               "baseline_ab": baseline_ab, "flipped": bool(flipped),
               "raw_judge_output": res["raw"], "prompt_version": res["prompt_version"],
               "temperature": 0.0, "timestamp": time.time()}

def run_experiment(items, judge_name, out_path, progress_every=5):
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out.exists():
        for line in open(out):
            try: done.add(json.loads(line)["item_id"])
            except Exception: pass
    if done:
        print(f"RESUMING: {len(done)} items already logged, skipping them")
    t0, n_items = time.time(), 0
    with open(out, "a") as f:
        for it in items:
            if it["item_id"] in done: continue
            for rec in run_item(it, judge_name):
                f.write(json.dumps(rec) + "\n")
            f.flush(); n_items += 1
            if n_items % progress_every == 0:
                rate = (time.time() - t0) / n_items
                remaining = (len(items) - len(done) - n_items) * rate / 60
                print(f"{n_items} items done | ~{rate:.0f}s/item | ~{remaining:.0f} min left")
    print("run complete ->", out)
    return str(out)

print("harness ready | factors:", list(_factor_levels()),
      "| calls/item:", 1 + 2**len(_factor_levels()))

harness ready | factors: ['position', 'formatting', 'length'] | calls/item: 9


In [7]:
#@title Cell 7 — PILOT: 15 items first (~15-20 min). Check flips before the full burn.
pilot = items[:15]
path = run_experiment(pilot, judge_name="qwen2.5-7b", out_path="results/raw_qwen.jsonl")

import pandas as pd
df = pd.DataFrame([json.loads(l) for l in open(path)])
print("UNPARSED rate:", (df.slot_verdict == "UNPARSED").mean().round(3))
print("TIE rate:", (df.slot_verdict == "TIE").mean().round(3))
print("flip rate overall:", df.flipped.mean().round(3))
print("flips by category:\n", df.groupby("category").flipped.mean().round(3))
# DECISION GATE: if close-pair flip rate is 0.0 here too, STOP and tell Claude -
# we tighten the close filter before burning 4 hours. If it's > 0, run Cell 8.

5 items done | ~11s/item | ~2 min left
10 items done | ~11s/item | ~1 min left
15 items done | ~11s/item | ~0 min left
run complete -> results/raw_qwen.jsonl
UNPARSED rate: 0.0
TIE rate: 0.0
flip rate overall: 0.458
flips by category:
 category
close    0.458
Name: flipped, dtype: float64


In [8]:
#@title Cell 8 — FULL RUN (~3-4.5 h for 300 items; resumable - rerun this cell after any disconnect)
path = run_experiment(items, judge_name="qwen2.5-7b", out_path="results/raw_qwen.jsonl")
# Colab free tier may disconnect. That is FINE: rerun Cell 1-6 quickly (model
# reload ~6 min), then rerun THIS cell - it resumes where it stopped.
# Save to Drive (Cell 10) every session, not just at the end.

RESUMING: 15 items already logged, skipping them
5 items done | ~10s/item | ~46 min left
10 items done | ~9s/item | ~43 min left
15 items done | ~9s/item | ~41 min left
20 items done | ~9s/item | ~41 min left
25 items done | ~9s/item | ~41 min left
30 items done | ~9s/item | ~40 min left
35 items done | ~10s/item | ~40 min left
40 items done | ~9s/item | ~39 min left
45 items done | ~9s/item | ~37 min left
50 items done | ~9s/item | ~35 min left
55 items done | ~9s/item | ~35 min left
60 items done | ~9s/item | ~33 min left
65 items done | ~9s/item | ~32 min left
70 items done | ~9s/item | ~30 min left
75 items done | ~8s/item | ~29 min left
80 items done | ~8s/item | ~28 min left
85 items done | ~8s/item | ~27 min left
90 items done | ~8s/item | ~26 min left
95 items done | ~8s/item | ~25 min left
100 items done | ~8s/item | ~25 min left
105 items done | ~8s/item | ~25 min left
110 items done | ~8s/item | ~24 min left
115 items done | ~8s/item | ~24 min left
120 items done | ~8s/item 

In [10]:
#@title Cell 9 — Analysis: flip rates, additivity, interaction model
import numpy as np, pandas as pd, json
df = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
FACTORS = [c for c in ["position", "formatting", "length"] if c in df.columns]

usable = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])].copy()
print(f"usable rows: {len(usable)}/{len(df)}")

def flip_rate_table(d):
    g = d.groupby(FACTORS)["flipped"].agg(["mean","sum","count"]).reset_index()
    g = g.rename(columns={"mean":"flip_rate","sum":"n_flips","count":"n"})
    g["pct"] = (g.flip_rate*100).round(1)
    return g

def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (round((c-h)*100,1), round((c+h)*100,1))

print("\n=== FLIP RATES: close pairs ===")
t = flip_rate_table(usable[usable.category == "close"])
t["ci"] = [wilson_ci(r.n_flips, r.n) for r in t.itertuples()]
print(t.to_string())
print("\n=== FLIP RATES: anchor pairs (expect ~0 - the robustness control) ===")
print(flip_rate_table(usable[usable.category == "anchor"]).to_string())

def additivity_check(d):
    t = flip_rate_table(d)
    def rate(**kw):
        m = t.copy()
        for f in FACTORS: m = m[m[f] == kw.get(f, 0)]
        return float(m.flip_rate.mean()) if len(m) else np.nan
    singles = {f: rate(**{f:1}) for f in FACTORS}
    all_on = rate(**{f:1 for f in FACTORS})
    s = sum(singles.values())
    verdict = ("super-additive" if all_on > s+0.02 else
               "antagonistic" if all_on < s-0.02 else "approximately additive")
    return {"singles": {k: round(v,3) for k,v in singles.items()},
            "sum_of_singles": round(s,3), "observed_compound": round(all_on,3),
            "verdict": verdict}
print("\n=== ADDITIVITY (close pairs) ===")
print(additivity_check(usable[usable.category == "close"]))

print("\n=== INTERACTION MODEL (close pairs, item-clustered SEs) ===")
import statsmodels.formula.api as smf
m = usable[usable.category == "close"].copy(); m["y"] = m.flipped.astype(int)
try:
    res = smf.logit("y ~ " + "*".join(FACTORS), data=m).fit(
        disp=False, cov_type="cluster", cov_kwds={"groups": m["item_id"]})
    tidy = pd.DataFrame({"coef": res.params, "p": res.pvalues})
    ci = res.conf_int(); tidy["ci_low"], tidy["ci_high"] = ci[0], ci[1]
    tidy["OR"] = np.exp(tidy.coef).round(3)
    print(tidy.round(3).to_string())
except Exception as e:
    print("model failed:", e)

usable rows: 2377/2400

=== FLIP RATES: close pairs ===
   position  formatting  length  flip_rate  n_flips    n   pct            ci
0         0           0       0   0.000000        0  260   0.0    (0.0, 1.5)
1         0           0       1   0.228346       58  254  22.8  (18.1, 28.4)
2         0           1       0   0.092664       24  259   9.3   (6.3, 13.4)
3         0           1       1   0.289062       74  256  28.9  (23.7, 34.7)
4         1           0       0   0.665385      173  260  66.5  (60.6, 72.0)
5         1           0       1   0.505929      128  253  50.6  (44.5, 56.7)
6         1           1       0   0.629344      163  259  62.9  (56.9, 68.6)
7         1           1       1   0.449219      115  256  44.9  (38.9, 51.0)

=== FLIP RATES: anchor pairs (expect ~0 - the robustness control) ===
   position  formatting  length  flip_rate  n_flips   n   pct
0         0           0       0      0.000        0  40   0.0
1         0           0       1      0.350       14  40 

/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [16]:
#@title Cell 9b — Corrected analysis (independence null + paired bootstrap)
import numpy as np, pandas as pd, json
df = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
FACTORS = ["position","formatting","length"]
u = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])].copy()

def cell_rate(d, pos, fmt, ln):
    m = d[(d.position==pos)&(d.formatting==fmt)&(d.length==ln)]
    return m.flipped.mean(), len(m)

def indep(*ps):
    q = 1.0
    for x in ps: q *= (1-x)
    return 1-q

def paired_bootstrap(d, cellA, cellB_parts, n_boot=2000, seed=0):
    """CI on (observed_compound - independence_prediction), resampling ITEMS."""
    rng = np.random.default_rng(seed)
    items = d.item_id.unique()
    piv = d.pivot_table(index="item_id", columns=FACTORS, values="flipped")
    diffs = []
    for _ in range(n_boot):
        samp = rng.choice(items, size=len(items), replace=True)
        s = piv.loc[samp]
        obs = s[cellA].mean()
        pred = indep(*[s[c].mean() for c in cellB_parts])
        diffs.append(obs - pred)
    return np.percentile(diffs, [2.5, 97.5])

for cat in ["close","anchor"]:
    d = u[u.category==cat]
    if len(d)==0: continue
    print(f"\n===== {cat.upper()} PAIRS (n items ~ {d.item_id.nunique()}) =====")
    pos,_ = cell_rate(d,1,0,0); fmt,_ = cell_rate(d,0,1,0); ln,_ = cell_rate(d,0,0,1)
    print(f"singles: position {pos*100:.1f}%  formatting {fmt*100:.1f}%  length {ln*100:.1f}%")
    combos = [("position x formatting",(1,1,0),[(1,0,0),(0,1,0)]),
              ("position x length",    (1,0,1),[(1,0,0),(0,0,1)]),
              ("formatting x length",  (0,1,1),[(0,1,0),(0,0,1)]),
              ("all three",            (1,1,1),[(1,0,0),(0,1,0),(0,0,1)])]
    print(f"{'combination':24s} {'pred':>7s} {'obs':>7s} {'diff':>8s}  {'95% CI (pp)':>18s}")
    for name, cell, parts in combos:
        obs,_ = cell_rate(d,*cell)
        pred = indep(*[cell_rate(d,*c)[0] for c in parts])
        try:
            lo,hi = paired_bootstrap(d, cell, parts)
            ci = f"[{lo*100:+.1f}, {hi*100:+.1f}]"
        except Exception as e:
            ci = "n/a"
        print(f"{name:24s} {pred*100:6.1f}% {obs*100:6.1f}% {(obs-pred)*100:+7.1f}  {ci:>18s}")

# slot-anchoring diagnostic (the paper's strongest single number)
print("\nslot-1 selection rate overall:", (u.slot_verdict=="1").mean().round(3))
print("baseline cell is a definitional zero (baseline vs itself) — report as such, exclude from models")


===== CLOSE PAIRS (n items ~ 260) =====
singles: position 66.5%  formatting 9.3%  length 22.8%
combination                 pred     obs     diff         95% CI (pp)
position x formatting      69.6%   62.9%    -6.7       [-10.8, -3.1]
position x length          74.2%   50.6%   -23.6      [-28.9, -18.2]
formatting x length        30.0%   28.9%    -1.1        [-5.8, +3.6]
all three                  76.6%   44.9%   -31.7      [-37.8, -25.1]

===== ANCHOR PAIRS (n items ~ 40) =====
singles: position 37.5%  formatting 10.0%  length 35.0%
combination                 pred     obs     diff         95% CI (pp)
position x formatting      43.8%   42.5%    -1.3        [-9.8, +8.3]
position x length          59.4%   32.5%   -26.9       [-43.5, -9.0]
formatting x length        41.5%   30.0%   -11.5       [-22.5, -1.3]
all three                  63.4%   30.0%   -33.4      [-47.9, -17.3]

slot-1 selection rate overall: 0.646
baseline cell is a definitional zero (baseline vs itself) — report as such, e

In [19]:
#@title Cell 9b — rerun with fresh reload (all three categories) latest
import numpy as np, pandas as pd, json
df = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
FACTORS = ["position","formatting","length"]
u = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])].copy()

def cell_rate(d, pos, fmt, ln):
    m = d[(d.position==pos)&(d.formatting==fmt)&(d.length==ln)]
    return m.flipped.mean(), len(m)

def indep(*ps):
    q = 1.0
    for x in ps: q *= (1-x)
    return 1-q

def paired_bootstrap(d, cellA, parts, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    piv = d.pivot_table(index="item_id", columns=FACTORS, values="flipped")
    items_ = piv.index.values
    diffs = []
    for _ in range(n_boot):
        s = piv.loc[rng.choice(items_, size=len(items_), replace=True)]
        diffs.append(s[cellA].mean() - indep(*[s[c].mean() for c in parts]))
    return np.percentile(diffs, [2.5, 97.5])

for cat in ["close", "anchor", "anchor2"]:
    d = u[u.category == cat]
    if len(d) == 0: continue
    print(f"\n===== {cat.upper()} (n items = {d.item_id.nunique()}) =====")
    pos,_ = cell_rate(d,1,0,0); fmt,_ = cell_rate(d,0,1,0); ln,_ = cell_rate(d,0,0,1)
    print(f"singles: position {pos*100:.1f}%  formatting {fmt*100:.1f}%  length {ln*100:.1f}%")
    for name, cell, parts in [
        ("position x formatting",(1,1,0),[(1,0,0),(0,1,0)]),
        ("position x length",    (1,0,1),[(1,0,0),(0,0,1)]),
        ("formatting x length",  (0,1,1),[(0,1,0),(0,0,1)]),
        ("all three",            (1,1,1),[(1,0,0),(0,1,0),(0,0,1)])]:
        obs,_ = cell_rate(d,*cell)
        pred = indep(*[cell_rate(d,*c)[0] for c in parts])
        lo,hi = paired_bootstrap(d, cell, parts)
        print(f"{name:24s} pred {pred*100:5.1f}%  obs {obs*100:5.1f}%  "
              f"diff {(obs-pred)*100:+6.1f}  CI [{lo*100:+.1f}, {hi*100:+.1f}]")

print("\nslot-1 selection rate overall:", round((u.slot_verdict=="1").mean(), 3))


===== CLOSE (n items = 260) =====
singles: position 66.5%  formatting 9.3%  length 22.8%
position x formatting    pred  69.6%  obs  62.9%  diff   -6.7  CI [-10.5, -2.7]
position x length        pred  74.2%  obs  50.6%  diff  -23.6  CI [-29.0, -18.1]
formatting x length      pred  30.0%  obs  28.9%  diff   -1.1  CI [-5.8, +3.9]
all three                pred  76.6%  obs  44.9%  diff  -31.7  CI [-37.9, -25.2]

===== ANCHOR (n items = 40) =====
singles: position 37.5%  formatting 10.0%  length 35.0%
position x formatting    pred  43.8%  obs  42.5%  diff   -1.3  CI [-9.8, +8.3]
position x length        pred  59.4%  obs  32.5%  diff  -26.9  CI [-43.5, -9.0]
formatting x length      pred  41.5%  obs  30.0%  diff  -11.5  CI [-22.5, -1.3]
all three                pred  63.4%  obs  30.0%  diff  -33.4  CI [-47.9, -17.3]

===== ANCHOR2 (n items = 40) =====
singles: position 35.0%  formatting 2.5%  length 22.5%
position x formatting    pred  36.6%  obs  35.0%  diff   -1.6  CI [-11.8, +8.4]
positio

In [14]:
print(final.groupby("category").n_votes.value_counts().unstack(fill_value=0))

n_votes     1   2   3  4  5
category                   
anchor     33   6   1  0  0
close     134  85  32  8  1


In [13]:
#@title Cell 9c — join human labels: n_votes check + accuracy per condition
from datasets import load_dataset
import pandas as pd, numpy as np, json

raw = load_dataset("lmsys/mt_bench_human_judgments")["human"].to_pandas()
raw = raw[raw["turn"] == 1]
def _ex(r):
    try:
        return pd.Series({"question": r["conversation_a"][0]["content"]})
    except Exception:
        return pd.Series({"question": None})
raw = pd.concat([raw.reset_index(drop=True),
                 raw.apply(_ex, axis=1).reset_index(drop=True)], axis=1).dropna(subset=["question"])

def human_label(g):
    v = g["winner"].tolist()
    na, nb, nt = (sum(x=="model_a" for x in v), sum(x=="model_b" for x in v),
                  sum(x=="tie" for x in v))
    lab = "A" if na > nb else ("B" if nb > na else "TIE")
    return pd.Series({"question": g.iloc[0]["question"], "human_label": lab,
                      "n_votes": len(v), "unanimous": (na == 0 or nb == 0) and nt == 0})

hl = (raw.groupby(["question_id","model_a","model_b"], as_index=False)
         .apply(human_label, include_groups=False).reset_index(drop=True))

df = pd.DataFrame([json.loads(l) for l in open("results/raw_qwen.jsonl")])
df = df.merge(hl[["question","human_label","n_votes","unanimous"]], on="question", how="left")
print("merge rate:", round(df.human_label.notna().mean(), 3))

# ---- THE OUTSTANDING CHECK: are anchors genuinely multi-vote unanimous? ----
print("\nn_votes distribution by category (per item):")
per_item = df.drop_duplicates("item_id")
print(per_item.groupby("category").n_votes.value_counts().unstack(fill_value=0))
print("\nanchors that are unanimous with >=2 votes:",
      int(((per_item.category=="anchor") & per_item.unanimous & (per_item.n_votes>=2)).sum()),
      "of", int((per_item.category=="anchor").sum()))

# ---- accuracy against human label, per condition ----
u = df[df.verdict_ab.isin(["A","B"]) & df.human_label.isin(["A","B"])]
acc = (u.groupby(["category","position","formatting","length"])
         .apply(lambda d: pd.Series({"acc": (d.verdict_ab==d.human_label).mean(), "n": len(d)}),
                include_groups=False).reset_index())
acc["acc_pct"] = (acc.acc*100).round(1)
for cat in acc.category.unique():
    print(f"\n=== ACCURACY vs HUMAN — {cat} (50% = chance) ===")
    print(acc[acc.category==cat][["position","formatting","length","acc_pct","n"]].to_string(index=False))

KeyError: 'question'

In [20]:
#@title Cell 10 — Save to Drive (RUN EVERY SESSION - the jsonl IS the paper)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/judge-compound-bias
!cp -v results/*.jsonl /content/drive/MyDrive/judge-compound-bias/
# To resume in a fresh runtime: after Cell 6, copy the file BACK before Cell 8:
# !mkdir -p results && cp /content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl results/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'results/raw_qwen.jsonl' -> '/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl'


In [21]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/judge-compound-bias
!cp -v results/*.jsonl /content/drive/MyDrive/judge-compound-bias/
!ls -la /content/drive/MyDrive/judge-compound-bias/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'results/raw_qwen.jsonl' -> '/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl'
total 1218
-rw------- 1 root root 207587 Aug 22 21:43 judge_compound_bias_day2.ipynb
-rw------- 1 root root 121360 Aug 22 20:16 judge_compound_bias.ipynb
-rw------- 1 root root 903573 Aug 22 21:43 raw_qwen.jsonl
-rw------- 1 root root   2585 Aug 23  2026 README.md
-rw------- 1 root root    108 Aug 23  2026 requirements.txt
-rw------- 1 root root   5295 Aug 23  2026 run_colab.py
drwx------ 2 root root   4096 Aug 22 19:43 src


## Decision gates in this notebook
1. **Cell 5**: length transform must be clean on real data (dirty=0). If not → paste output to Claude.
2. **Cell 7 (pilot)**: close-pair flip rate must be > 0. If 0 → stop, tighten close filter with Claude before the full burn.
3. **Cell 8**: resumable — disconnects cost nothing if you saved to Drive.

Day 3+ = full run finishes → Cell 9 analysis is the paper's Figure 1 + Table 1 → second judge (rerun Cells 6-8 with a different MODEL_NAME and out_path) only if everything is clean.

In [6]:
items_all = items + strong_anchor_items
print(len(items_all), "items")   # expect 340

!mkdir -p results
!cp /content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl results/
!ls -la results/

340 items
cp: cannot stat '/content/drive/MyDrive/judge-compound-bias/raw_qwen.jsonl': No such file or directory
total 8
drwxr-xr-x 2 root root 4096 Aug 22 23:10 .
drwxr-xr-x 1 root root 4096 Aug 22 23:10 ..
